# 02 · Consultas, arrays, agregação e paginação

**Pré-requisito:** notebooks 00 e 01.

**Nesta etapa:** perguntar coisas ao banco. No fim, o **Exercício 2** (itens a até e).

In [ ]:
%pip install -q "pymongo[srv]" python-dotenv pillow numpy matplotlib requests

## Conexão com o seu banco no Atlas

A célula abaixo lê a sua URI do arquivo **`.env`** da pasta (criado no notebook 00, Etapa 6) e testa a conexão.

In [ ]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()                                   # lê o arquivo .env da pasta
client = MongoClient(os.environ["MONGO_URI"])
client.admin.command("ping")
print("Conectado ao MongoDB Atlas")

Para todos terem o mesmo resultado, vamos recriar a coleção com os 8 alunos da aula:

In [ ]:
ALUNOS = [
    {"nome": "Ana",     "idade": 22, "curso": "Computação"},
    {"nome": "Bruno",   "idade": 19, "curso": "Medicina"},
    {"nome": "Carla",   "idade": 24, "curso": "Direito"},
    {"nome": "Matheus", "idade": 21, "curso": "Computação", "disciplinas": ["Python"]},
    {"nome": "Mariana", "idade": 26, "curso": "Engenharia"},
    {"nome": "Pedro",   "idade": 20, "curso": "Medicina"},
    {"nome": "Juliana", "idade": 23, "curso": "Ciência de Dados"},
    {"nome": "Lucas",   "idade": 18, "curso": "Direito"},
]

alunos = client["escola"]["alunos"]
alunos.delete_many({})
alunos.insert_many(ALUNOS)

def mostrar(filtro, projecao=None):
    projecao = projecao or {"_id": 0, "nome": 1, "idade": 1, "curso": 1}
    for a in alunos.find(filtro, projecao):
        print(a)
    print("-" * 50)

print(alunos.count_documents({}), "alunos")

---
## 1. Uma consulta também é um documento

| SQL | MongoDB |
|---|---|
| `WHERE idade > 21` | `{"idade": {"$gt": 21}}` |

### Operadores de comparação

| Operador | Significado |
|---|---|
| `$eq` / `$ne` | igual / diferente |
| `$gt` / `$gte` | maior / maior ou igual |
| `$lt` / `$lte` | menor / menor ou igual |
| `$in` / `$nin` | está / não está na lista |

In [ ]:
mostrar({"idade": {"$gt": 21}})
mostrar({"idade": {"$gte": 20, "$lte": 23}})            # dois operadores no mesmo campo = E
mostrar({"curso": {"$in": ["Direito", "Medicina"]}})
mostrar({"curso": {"$nin": ["Direito", "Medicina"]}})

### Operadores lógicos

| Operador | Significado |
|---|---|
| `$and` | todas as condições |
| `$or` | pelo menos uma |
| `$nor` | nenhuma delas |
| `$not` | nega uma condição |

In [ ]:
condicoes = [{"idade": {"$lt": 21}}, {"curso": "Direito"}]
print("$and:"); mostrar({"$and": condicoes})
print("$or:");  mostrar({"$or": condicoes})
print("$nor:"); mostrar({"$nor": condicoes})
print("$not:"); mostrar({"idade": {"$not": {"$gt": 21}}})

### Existência, tipo e texto

| Operador | Significado |
|---|---|
| `$exists` | o campo existe neste documento? |
| `$type` | o campo é de um certo tipo? |
| `$regex` | o texto segue um padrão? (`^M` = começa com M) |

In [ ]:
mostrar({"disciplinas": {"$exists": True}})
mostrar({"idade": {"$type": "int"}})
mostrar({"nome": {"$regex": "^M"}})
mostrar({"nome": {"$regex": "^m", "$options": "i"}})   # "i" ignora maiúsculas/minúsculas

---
## 2. Atualizando campos e listas

| Operador | O que faz |
|---|---|
| `$set` / `$unset` | define / remove um campo |
| `$inc` / `$mul` | soma / multiplica |
| `$rename` | renomeia um campo |
| `$min` / `$max` | só troca se o novo valor for menor / maior |
| `$currentDate` | grava a data de agora |
| `$push` | adiciona ao fim da lista (mesmo se já existir) |
| `$addToSet` | adiciona só se ainda não existir |
| `$each` | vários itens de uma vez (com `$push` ou `$addToSet`) |
| `$pop` | remove o último (`1`) ou o primeiro (`-1`) |
| `$pull` | remove um valor específico |

In [ ]:
m = {"nome": "Matheus"}
ver = lambda: print(alunos.find_one(m, {"_id": 0, "disciplinas": 1}))

alunos.update_one(m, {"$push": {"disciplinas": "SQL"}});        ver()
alunos.update_one(m, {"$addToSet": {"disciplinas": "SQL"}});    ver()   # já existe: nada muda
alunos.update_one(m, {"$push": {"disciplinas": {"$each": ["Estatística", "IA"]}}}); ver()
alunos.update_one(m, {"$pop": {"disciplinas": -1}});            ver()   # tira o primeiro
alunos.update_one(m, {"$pull": {"disciplinas": "SQL"}});        ver()

In [ ]:
alunos.update_one({"nome": "Pedro"}, {"$inc": {"idade": 1}})            # aniversário
alunos.update_many({}, {"$currentDate": {"atualizado_em": True}})
print(alunos.find_one({"nome": "Pedro"}, {"_id": 0}))

# desfaz para os próximos exemplos
alunos.update_one({"nome": "Pedro"}, {"$inc": {"idade": -1}})
alunos.update_many({}, {"$unset": {"atualizado_em": ""}})

---
## 3. Agregação: a esteira de fábrica

Um **pipeline** é uma lista de estágios. Os documentos entram no primeiro, e cada estágio trabalha sobre o resultado do anterior.

| Estágio | Em SQL | O que faz |
|---|---|---|
| `$match` | `WHERE` | filtra |
| `$group` | `GROUP BY` | agrupa e calcula `$sum`, `$avg`, `$min`, `$max` |
| `$project` | `SELECT` | escolhe e cria campos |
| `$sort` | `ORDER BY` | ordena (`1` cresc., `-1` decresc.) |
| `$skip` / `$limit` | `OFFSET` / `LIMIT` | pula / limita |

No `$group`, o campo de agrupamento vai em `_id`, e **o nome de um campo precisa de `$` na frente** (`"$curso"`) para virar o *valor* do campo.

In [ ]:
pipeline = [
    {"$match": {"idade": {"$gte": 20}}},
    {"$group": {"_id": "$curso", "total": {"$sum": 1}, "media": {"$avg": "$idade"}}},
    {"$sort": {"media": -1}},
]
for linha in alunos.aggregate(pipeline):
    print(linha)

In [ ]:
# $project: renomear, esconder e calcular campos
for linha in alunos.aggregate([
    {"$group": {"_id": "$curso", "media": {"$avg": "$idade"}}},
    {"$project": {"_id": 0, "curso": "$_id", "media": {"$round": ["$media", 1]}}},
]):
    print(linha)

---
## 4. Paginação com `skip` + `limit`

Página *p* com *n* itens: pule `(p − 1) × n`, pegue `n`.

In [ ]:
por_pagina = 3
total = alunos.count_documents({})
paginas = (total + por_pagina - 1) // por_pagina     # divisão arredondando para cima

for p in range(1, paginas + 1):
    print(f"--- Página {p} ---")
    for a in alunos.find({}, {"_id": 0, "nome": 1}).sort("nome", 1).skip((p - 1) * por_pagina).limit(por_pagina):
        print(a["nome"])

---
## Agora vamos fazer o Exercício 2 · Consultas da secretaria

**Você vai usar:** `aggregate`, `$group`, `$sort`, `$or`/`$and`, `$regex`, `$push` + `$each`, `$addToSet`, `$pop`, `skip`/`limit`, `$multiply`.

Antes de começar, rode a célula abaixo para garantir os 8 alunos originais.

In [ ]:
alunos.delete_many({})
alunos.insert_many([dict(a) for a in ALUNOS])
print(alunos.count_documents({}), "alunos prontos")

**a) Estatísticas por curso.** Para cada curso: idade **média**, **mínima**, **máxima** e **total** de alunos, da maior média para a menor.

*Resultado esperado (ordem):* Engenharia (26), Ciência de Dados (23), Computação (21,5), Direito (21), Medicina (19,5).

In [ ]:
# seu código aqui

**b) Filtro combinado.** Alunos que (têm idade **entre 20 e 25** **e** curso **diferente de Medicina**) **ou** cujo nome começa com **“M”**.

*Resultado esperado:* Ana, Carla, Matheus, Mariana, Juliana.

In [ ]:
# seu código aqui

**c) Arrays.**
1. Adicione `["Banco de Dados", "Estatística"]` a **todos** os alunos (`$push` + `$each`).
2. Adicione `"Machine Learning"` só onde ainda **não** existir (`$addToSet`).
3. Remova a **primeira** disciplina de cada lista (`$pop`).
4. Mostre `nome` e `disciplinas`.

*Dica:* `update_many({}, ...)` aplica a todos. Para o Matheus, o resultado esperado é `["Banco de Dados", "Estatística", "Machine Learning"]`.

In [ ]:
# seu código aqui

**d) Paginação.** Mostre **3 alunos por página**, só `nome` e `curso`, ordenados pela **idade crescente**, com o número da página antes de cada grupo.

*Resultado esperado na página 1:* Lucas, Bruno, Pedro.

In [ ]:
# seu código aqui

**e) Campo calculado.** Para cursos cujo nome contém a letra **“a”**, mostre a média de idade e um campo `idade_media_meses` (média × 12, use `$multiply`), do **menor** para o **maior**.

*Resultado esperado:* Medicina (234), Computação (258), Ciência de Dados (276), Engenharia (312). Direito não tem “a”!

In [ ]:
# seu código aqui

**Próximo:** `03_Conexao_Segura_e_App_Streamlit.ipynb`